# Módulo 3: Ferramentas — Código e Navegador na Veia

![Overview](../shared/img/03.drawio.png)

Neste módulo, você vai dar um upgrade gigantesco na Aria plugando as duas **ferramentas gerenciadas do AgentCore**.

## O que você vai aprender

- **Ferramentas Gerenciadas** — serviços da AWS já prontinhos pra você encaixar no agente com apenas duas linhas de código
- **Interpretador de Código** — Uma maquininha isolada para ela programar em Python, calcular com perfeição, gerar gráficos e limpar arquivos.
- **Navegador Web** — um navegador Chrome sem tela gerenciado pela AWS pra Aria navegar e extrair informações.

No final do módulo, a Aria conseguirá rodar código Python perfeito e pesquisar na internet real — e você não vai precisar abrir nenhum servidor pra isso.

---
## Atualizando as dependências

In [ ]:
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
import sys; sys.path.insert(0, '..')
# Importa as ferramentas compartilhadas do workshop.
from shared.ensure_ready import ensure_ready

# Verifica se o Runtime do módulo anterior está rodando (necessário para este módulo).
config = ensure_ready("03")

---
## O que mudou de lá pra cá?

No módulo anterior, o código do agente era super básico. Agora a gente encaixou as ferramentas nela. Se liga nas novidades:

### O Interpretador de Código

- Ele roda scripts em Python dentro de um **contêiner isolado e seguro**.
- Consegue analisar planilhas de dados, calcular fórmulas matemáticas absurdas, gerar imagens/gráficos e lidar com arquivos pesados.
- A caixa de segurança onde o código roda **não tem acesso à internet** (justamente pra evitar scripts maliciosos de fugirem).
- É perfeito se você pedir: 'calcula o juro composto desse investimento' ou 'desenha um gráfico desses números aqui'.

### Navegador Web

- **Um Chrome invisível** rodando na AWS
- Capaz de acessar links, roubar o conteúdo puro da página, preencher dados ou bater fotos das telas.
- Dá acesso real e imediato ao que tá acontecendo no mundo hoje.
- Perfeito para perguntar: 'qual o preço do Bitcoin hoje às 14h' ou 'lê aquele artigo ali no blog'.

### Serviço Premium Gerenciado

Você não vai instalar Chrome no seu Linux, não vai configurar docker pra Python nem se preocupar com segurança. O AgentCore toma conta de toda essa dor de cabeça e te entrega um objetozinho de Python simples pra você ligar no agente.

Para ver o código completinho, abra o arquivo [agent/main.py](agent/main.py) em outra aba.

---
## Sobe a nova versão! (Deploy)

Vamos mandar esse código novo pro mesmo Runtime de antes. O AgentCore substitui a versão atual automaticamente de forma suave e transparente.

In [ ]:
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
import sys; sys.path.insert(0, '..')
# Importa as ferramentas compartilhadas do workshop.
from shared import deploy_agent

# Módulo de deploy — vai atualizar o Runtime com o novo código que inclui ferramentas.
result = deploy_agent.deploy(
    agent_dir="agent",
    runtime_name="aria_agent",
)

runtime_arn = result["runtime_arn"]
print(f"\nRuntime ARN: {runtime_arn}")

---
## Enable Tracing for Code Interpreter and Browser Tool

Enable **Tracing** on both managed tools so that tool invocations appear in the AgentCore Observability dashboard.

### Code Interpreter

1. Open the **Amazon Bedrock AgentCore** console and navigate to **Built-in tools** > **Code Interpreter**
2. Scroll down to the **Tracing** section and click **Edit**

![Tracing section](../shared/img/tracing-01.png)

3. Toggle **Enable** on and click **Save**

![Enable tracing](../shared/img/tracing-02.png)

### Browser Tool

1. Navigate to **Built-in tools** > **Browser**
2. Scroll down to the **Tracing** section and click **Edit**
3. Toggle **Enable** on and click **Save**

---
## Configurando a chamada do agente

Vamos criar a rotina pra testar de novo. Dessa vez montei um helper local pra não poluir tanto as caixinhas de código. Você pode ver exatamente como as chamadas são feitas no `invoke_agent_runtime`.

In [ ]:
import boto3, json, uuid
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
import sys; sys.path.insert(0, '..')
# Importa as ferramentas compartilhadas do workshop.
from shared import utils

client = boto3.client("bedrock-agentcore", region_name="us-east-1")

# Testa o agente enviando prompts que devem acionar as ferramentas.
def invoke(prompt):
    """Simple wrapper to invoke and stream. Uses a fresh session each time
    # Navegador Web: abre um navegador headless para buscar informações na internet.
    to avoid a known issue where the browser tool fails on reuse within
    the same runtime session."""
    session_id = str(uuid.uuid4())
    response = client.invoke_agent_runtime(
        agentRuntimeArn=runtime_arn,
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}).encode(),
    )
    return utils.stream_sse_response(response["response"])

---
## Testando a Mente Matemática

No módulo 2 a Aria chutava números nas respostas. Agora, se ela não souber, ela vai **escrever um programa em Python pra fazer a conta de verdade**.

### Cálculo de Juros Compostos

Isso é legal porque exige matemática exata, não apenas chutar um palpite.

In [ ]:
# Code Interpreter: sandboxed Python execution
# Testa o agente enviando prompts que devem acionar as ferramentas.
invoke("Calculate compound interest on $10,000 at 7% annually for 30 years")

---
## Testando o Acesso à Internet

O navegador deixa a Aria checar coisas que estão rolando na internet no mundo real. Vamos mandar uma pergunta do presente.

### Checando informações vivas

In [ ]:
# Browser Tool: headless Chrome managed by AgentCore
# Testa o agente enviando prompts que devem acionar as ferramentas.
invoke("What is the top headline on Hacker News?")

---
## Quer se Aprofundar?

Pra entender de verdade os detalhes profundos das ferramentas que você acabou de ligar:

- **Code Interpreter:** [Veja a documentação](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/code-interpreter-tool.html) — Mostra como ler arquivos da máquina segura, os limites de espaço e como adicionar bibliotecas no sandbox.
- **Navegador:** [Documentação aqui](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-tool.html) — Tem detalhes incríveis sobre como gerenciar cookies de sessão, perfis para preencher formulários de login na web sozinhos e etc.
- **Strands e as ferramentas:** [Repositório Aberto SDK](https://github.com/strands-agents/sdk-python) — mostram como você pode criar ferramentas personalizadas no seu código Python com o framework do Strands.

---
## O que vem a seguir

Agora o negócio ficou de verdade! A Aria tem uma super inteligência computacional e enxerga toda a internet. Só tem um problema: ela continua desmemoriada e esquece a conversa 5 minutos depois que você fecha o terminal.

No **Módulo 4: Memória**, vamos curar essa amnésia dando uma memória persistente pra ela:
- Lembrar de você em conversas diferentes ('Oi Aria, lembra que semana passada te falei que eu trabalho na área de RH?')
- Construir familiaridade com a sua forma de trabalhar com o tempo
- Fornecer um tratamento que fica cada vez mais refinado a cada interação de chat.

---
## Record progress

In [ ]:
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
import sys; sys.path.insert(0, '..')
# Importa as ferramentas compartilhadas do workshop.
from shared import progress

progress.show("03")

---

**Próximo Passo: [Módulo 4 -- Dando Memória Pra Valer pra Aria](../04-memory/notebook.ipynb)**